In [ ]:
# Calculate historical cumulative conflict averages.
import pandas as pd  
df1 = pd.read_excel(r"E:\AExperiment\11Water\E_RF_Data\B_Clustered_panel\03_Clustered_panel_1119.xlsx")  
df1 = df1.sort_values(by=['Country', 'year'])  
df1['Average_conflict'] = (  
    df1.groupby('Country')['Conflict_count_yr'].cumsum() /   
    (df1.groupby('Country').cumcount() + 1)  
)  
#df1.to_excel('C:\\Users\\A\\Desktop\\03_Clustered_panel_1119.xlsx', index=False) 
#print(df1)  


In [ ]:
# Append SSP5 data; map 2019 conflict averages to 2050.
import pandas as pd  
df2 = pd.read_excel("E:\\AExperiment\\11Water\\E_RF_Data\\E_ssp_predication\\ssp1-5_future_data.xlsx", sheet_name='ssp5')  
#df2 = pd.read_excel(r"E:\\AExperiment\\11Water\\E_RF_Data\\E_ssp_predication\\ssp1-5_future_data.xlsx", sheet_name='ssp1') 
df2 = df2[df2['year'] == 2050]  
df2.rename(columns={  
    'country': 'Country',
    'country_code': 'ccode_iso3',
    'population_density_ssp5': 'population_density', 
    #'population_density_ssp1': 'population_density', 
    'GDP per capita_ssp5(constant 2015 US$)': 'gdp_percapita',  
    #'GDP per capita_ssp1(constant 2015 US$)': 'gdp_percapita', 
    'rural_population_ssp5(%)': 'rural_population',
    #'rural_population_ssp1(%)': 'rural_population',
    'precipitation_ssp5': 'precipitation',  
    #'precipitation_ssp1': 'precipitation',  
    'bws_ssp5': 'bws_score'}, inplace=True)  
    #'bws_ssp1': 'bws_score'}, inplace=True)  
df = pd.concat([df1, df2], ignore_index=True)  
conflict_2019 = df.loc[df['year'] == 2019, ['ccode_iso3', 'Average_conflict']]
conflict_2019_unique = conflict_2019.drop_duplicates(subset='ccode_iso3')
conflict_2019_map = conflict_2019_unique.set_index('ccode_iso3')['Average_conflict']
df.loc[df['year'] == 2050, 'Average_conflict'] = df.loc[df['year'] == 2050, 'ccode_iso3'].map(conflict_2019_map).fillna(df.loc[df['year'] == 2050, 'Average_conflict'])
#df.to_excel('C:\\Users\\A\\Desktop\\output.xlsx', index=False)


In [ ]:
# Preprocess and standardize predictors.
import numpy as np  
from sklearn.preprocessing import StandardScaler
label = ['Joint Data-Sharing Systems','Cross-Border Basin Agreements','Transboundary Water Quality Standards',
         'Coordinated Hydropower Operations','Joint Water Allocation Models','Collaborative River Basin Planning']  
SEdata = ['population_density', 'export', 'gdp_percapita', 'rural_population', 'precipitation', 'bws_score']
df.loc[df['Both'] == 1, ['Conflict', 'Cooperation']] = 1  
df['Cooperation'] = df[label].any(axis=1).astype(int)
df['potential_Conflict'] = df['Conflict'] 
df = df[['Country', 'ccode_iso3', 'Conflict', 'Average_conflict', 'year'] + SEdata]
pd.set_option('future.no_silent_downcasting', True)  
df = df.replace('..', np.nan)  
scaler = StandardScaler()  
df[SEdata] = scaler.fit_transform(df[SEdata])  
df = df.dropna(subset=df.columns.difference(['Conflict']))
num_rows = df.shape[0]  
print("Number of rows:", num_rows)  


In [ ]:
# Define target and temporal training/test sets.
y = 'Conflict'
x = df[['Average_conflict'] + SEdata] 
cols_model = ['Country', 'ccode_iso3', 'year', 'Average_conflict'] + SEdata
df_model = df[cols_model]
mask_train = df['year'] <= 2019  
x_train = df_model.loc[mask_train, ['Average_conflict'] + SEdata]
y_train = df.loc[mask_train, y]
mask_test = df['year'] > 2019
x_test = df_model.loc[mask_test].copy()
y_test = df.loc[mask_test, y]
x_test_features = x_test[['Average_conflict'] + SEdata]
#mask_combined = (~mask_train) & (df['potential_Conflict'] == 1)  
#x_test_2010 = x.loc[mask_combined]   # actual group  
#y_test_2010 = df.loc[mask_combined, y] 


In [ ]:
# Fit random forest classifier and predict probabilities.
from sklearn.ensemble import RandomForestClassifier      
from sklearn.metrics import classification_report, accuracy_score      
clf = RandomForestClassifier(random_state=42)    
clf.fit(x_train, y_train)        
y_pred = clf.predict(x_test_features)
y_prob = clf.predict_proba(x_test_features)
for idx, (pred, prob) in enumerate(zip(y_pred, y_prob)):  
    print(f"Sample {idx + 1}: Predicted = {pred}, Probability of 1 = {prob[1]:.4f}")  


In [ ]:
# Export predictions.
x_test['Conflict_Probability'] = y_prob[:, 1]
x_test.to_excel(r"C:\Users\Lenovo\Desktop\Clustered_Conflict_Probability_SSP5.xlsx", index=False)


In [ ]:
# Hierarchical clustering of predicted probabilities.
import pandas as pd  
import matplotlib.pyplot as plt  
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster  
from scipy.spatial.distance import pdist  
df_prob = pd.read_excel(r"C:\Users\Lenovo\Desktop\Clustered_Conflict_Probability_SSP5.xlsx")  
data = df_prob['Conflict_Probability'].values.reshape(-1, 1)  
Y = pdist(data, 'euclidean')  
Z = linkage(Y, 'ward')  
plt.figure(figsize=(10, 7))  
dendrogram(Z)  
plt.title('Hierarchical Clustering Dendrogram')  
plt.xlabel('Sample index')  
plt.ylabel('Distance')  
plt.show()  
max_distance = max(Z[:, 2])  
distance_threshold = max_distance / 2  
clusters = fcluster(Z, distance_threshold, criterion='distance')  
df_prob['Cluster'] = clusters  
print(df_prob) 
df_prob.to_excel(r"C:\Users\Lenovo\Desktop\Clustered_Conflict_Probability_241119_SSP5.xlsx", index=False)
